# Verificación de Datos - ShopNow
---

### ¿Qué hace este cuaderno?

Están preparadas **cinco (05) pruebas automáticas** para revisar que los datos subidos están bien y no tienen fallos poco convenientes antes de empezar con los análisis. 

**¿Qué estamos vigilando?**
*   **Dinero:** Como que no haya productos con precio 0 o con precios negaivos (queries para comprobar que el filtrado de datos sintéticos se ha configurado de manera correcta)
*   **Lógica:** Que haya conexiones entre tablas eficientes.
*   **Limpieza:** Que no se nos cuelen clientes duplicados por error.

---

### Cómo usarlo
Asegrarse de que el archivo de credenciales tc-novarix-data.json esté subido en la carpeta raíz del proyecto. Ejecutar la CELDA DE CONEXIÓN para validar el acceso al proyecto tc-novarix-data.
**Resultados:** Si sale el mensaje de que no se han encontrado errores, es que todo está perfecto.

In [1]:
# CELDA DE CONEXIÓN
import os
from google.cloud import bigquery

# CONFIGURACIÓN MANUAL
NOMBRE_ARCHIVO_JSON = "tc-novarix-data.json"
ID_PROYECTO = "tc-novarix-data" 

project_id = "tc-novarix-data" 
dataset_id = "shopnow"

# Ruta
ruta_raiz = os.path.dirname(os.getcwd())
ruta_final_json = os.path.join(ruta_raiz, NOMBRE_ARCHIVO_JSON)

# Variable de entorno de Google directamente
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = ruta_final_json

print(f"Buscando el archivo en: {ruta_final_json}")

if os.path.exists(ruta_final_json):
    print("Se ha encontrado el archivo JSON")
    try:
        client = bigquery.Client(project=ID_PROYECTO)
        print(f"¡CONECTADO al proyecto {ID_PROYECTO}!")
    except Exception as e:
        print(f"Error al conectar con BigQuery: {e}")
else:
    print("SIGUE SIN ENCONTRARSE.")
    print(f"Contenido de la carpeta raíz: {os.listdir(ruta_raiz)}")

# importación para evitar que me salgan warnings que me "ensucien" el notebook
import warnings
warnings.filterwarnings('ignore')

Buscando el archivo en: c:\Users\maria\Documents\My bootcamp\GitHub\DS-Online-Maria-Rodriguez\Team_Challenges\TC_02_Sprint_06_SQL\TC-Novarix-Data\tc-novarix-data.json
Se ha encontrado el archivo JSON
¡CONECTADO al proyecto tc-novarix-data!


In [2]:
# Query de Verificación №1: ¿Quiénes son los TOP 10 clientes?

query_clientes_vip = f"""
SELECT 
    c.customer_id,
    c.email,
    COUNT(DISTINCT o.order_id) AS total_pedidos,
    ROUND(SUM(oi.unit_price * oi.quantity), 2) AS gasto_total
FROM `{project_id}.{dataset_id}.customers` AS c
JOIN `{project_id}.{dataset_id}.orders` AS o 
    ON c.customer_id = o.customer_id
JOIN `{project_id}.{dataset_id}.order_item_id` AS oi 
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.email
ORDER BY gasto_total DESC
LIMIT 10
"""

print("Identificando a los mejores clientes...")

try:
    df_vip = client.query(query_clientes_vip).to_dataframe()
    if df_vip.empty:
        print("No se han podido cruzar los datos de clientes con ventas.")
    else:
        print(f"Top 10 Clientes VIP:")
        print(df_vip)
except Exception as e:
    print(f"Error: {e}")

# Esta query es una prueba de la integridad de referencias entre clientes - pedidos - detalles.

Identificando a los mejores clientes...
Top 10 Clientes VIP:
   customer_id                        email  total_pedidos  gasto_total
0          278  vendrellrodrigo@example.com              8      6292.58
1          414           theo04@example.com              7      5626.69
2           86        cristal80@example.com              7      4950.09
3          374            zdiaz@example.org              8      4553.25
4          459        william77@example.org              8      4527.74
5          395            yan41@example.net             11      4281.37
6          165   jose-antonio51@example.com              5      4261.57
7          208  mcdonaldcharles@example.org              6      4231.49
8          297   sophieda-costa@example.com              4      4097.86
9          225  kristinacarlson@example.com              8      4052.82


In [3]:
# Query de Verificación Nº2: ¿Hay clientes con el mismo email?

query_duplicados = f"""
SELECT 
    email, 
    COUNT(*) as total_repeticiones
FROM `{project_id}.{dataset_id}.customers`
GROUP BY email
HAVING total_repeticiones > 1
ORDER BY total_repeticiones DESC
"""

print("Buscando duplicados en la tabla de clientes...")

# Ejecución de la query
try:
    df_duplicados = client.query(query_duplicados).to_dataframe()

    if df_duplicados.empty:
        print("No se han encontrado emails duplicados.")
    else:
        print(f"Se han encontrado un total de {len(df_duplicados)} emails repetidos:")
        print(df_duplicados)
except NameError:
    print("Error: Primero tienes que ejecutar la celda de conexión con el JSON.")
except Exception as e:
    print(f"Error: {e}")

# Aquí bscamos  que el DE haya filtrado bien los datos sintéticos.

Buscando duplicados en la tabla de clientes...
No se han encontrado emails duplicados.


In [5]:
# Query de Verificación №3: ¿Cuál es el volumen de facturación según el estado del pedido?

query_estado_pedidos = f"""
SELECT 
    o.status AS estado,
    COUNT(DISTINCT o.order_id) AS numero_pedidos,
    ROUND(SUM(oi.unit_price * oi.quantity), 2) AS facturacion_total,
    ROUND(AVG(oi.unit_price * oi.quantity), 2) AS ticket_medio
FROM `{project_id}.{dataset_id}.orders` AS o
JOIN `{project_id}.{dataset_id}.order_item_id` AS oi 
    ON o.order_id = oi.order_id
GROUP BY estado
ORDER BY facturacion_total DESC
"""

print("Analizando volumen de facturación...")

try:
    df_estado = client.query(query_estado_pedidos).to_dataframe()
    if df_estado.empty:
        print("No hay datos de pedidos para analizar.")
    else:
        print(f"Estado ventas:")
        print(df_estado)
except Exception as e:
    print(f" Error: {e}")

# Verifica que la columna 'status' de orders es consistente con las ventas.

Analizando volumen de facturación...
Estado ventas:
       estado  numero_pedidos  facturacion_total  ticket_medio
0   entregado            1180          498025.17        139.07
1     enviado             313          123798.96        144.29
2  en_proceso             203           84776.51        144.92
3   cancelado             136           58240.24        139.66
4   pendiente             104           50445.88        156.66
5    devuelto              64           33725.25        165.32


In [6]:
# Query de Verificación №4: ¿Hay productos con precio igual a 0 o precios negativos?

query_precios_falsos = f"""
SELECT 
    product_id,
    name,
    price
FROM `{project_id}.{dataset_id}.products`
WHERE price <= 0
ORDER BY price ASC
"""

print("Buscando errores en los precios...")

# Ejecución de la query
try:
    df_precios = client.query(query_precios_falsos).to_dataframe()

    if df_precios.empty:
        print("Todos los productos tienen un precio correcto, superior a 0€.")
    else:
        print(f"Error importante. Se han detectado {len(df_precios)} productos con precio incorrecto:")
        print(df_precios)

except NameError:
    print("Error: Primero tienes que ejecutar la celda de conexión con el JSON.")
except Exception as e:
    print(f"Error: {e}")

# Esta prueba asegura que todos los productos tengan un precio 
# válido para evitar ventas a 0€ o errores en el catálogo.

Buscando errores en los precios...
Todos los productos tienen un precio correcto, superior a 0€.


In [7]:
# Query de Verificación №5: ¿Cuáles son los ingresos totales por mes?

query_ingresos_mes = f"""
SELECT 
    FORMAT_DATE('%Y-%m', o.ordered_at) AS mes,
    ROUND(SUM(oi.unit_price * oi.quantity), 2) AS ingresos_totales,
    COUNT(DISTINCT o.order_id) AS total_pedidos
FROM `{project_id}.{dataset_id}.orders` AS o
JOIN `{project_id}.{dataset_id}.order_item_id` AS oi 
    ON o.order_id = oi.order_id
GROUP BY mes
ORDER BY mes DESC
"""

print("Calculando ingresos mensuales...")

try:
    df_ingresos = client.query(query_ingresos_mes).to_dataframe()
    if df_ingresos.empty:
        print("No se han encontrado datos de ventas.")
    else:
        print(f"Resultados de facturación mensual:")
        print(df_ingresos)
except Exception as e:
    print(f"Error: {e}")

# Esta query confirma que la relación entre orders y order_items es correcta.

Calculando ingresos mensuales...
Resultados de facturación mensual:
        mes  ingresos_totales  total_pedidos
0   2025-04          36690.34             69
1   2025-03          30189.53             68
2   2025-02          19989.62             59
3   2025-01          28134.98             71
4   2024-12          29508.06             88
5   2024-11          29764.20             74
6   2024-10          34227.06             74
7   2024-09          36333.98             86
8   2024-08          37444.01             73
9   2024-07          26178.56             68
10  2024-06          34629.73             79
11  2024-05          40738.73             83
12  2024-04          34764.46             87
13  2024-03          31095.72             63
14  2024-02          28047.04             71
15  2024-01          22603.69             59
16  2023-12          29438.61             57
17  2023-11          28288.33             74
18  2023-10          29701.13             71
19  2023-09          24074.25   

## Fin de la Verificación
Si todos los bloques anteriores han mostrado el mensaje de salida correctamente, los datos están listos para la fase de análisis y visualización.